# Spine 3 · Graded

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnakhoa/llm-engineering-playground/blob/main/spine/03-graded/graded.ipynb)

Your Flagship Agent gets graded: a suite of Cases over every Action it has,
Checks that assert on Backend state and on what the agent *tried*, and a
judge rubric for what state cannot show. You will catch a regression that
leaves the Backend looking perfect, then turn an incident into a Case of
your own.

Everything here runs **Offline**: replayed from reviewed recordings, no API key.
The lesson that goes with this notebook is `spine/03-graded/README.md`.

In [ ]:
# Setup. Locally this finds the course folder. On Colab it clones the course
# into the current folder once, and finds that clone again when you re-run this
# cell or restart the runtime.
import subprocess
import sys
from pathlib import Path

COURSE_URL = "https://github.com/dnakhoa/llm-engineering-playground.git"
CLONE = Path.cwd() / "llm-engineering-playground"


def find_course(start):
    for folder in [start, *start.parents, CLONE]:
        if (folder / "company").is_dir() and (folder / "llm").is_dir():
            return folder
    return None


COURSE = find_course(Path.cwd())
if COURSE is None:
    subprocess.run(["git", "clone", "--depth", "1", COURSE_URL, str(CLONE)], check=True)
    COURSE = CLONE
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("Course at", COURSE)

## 1. The Graded suite

Four Cases cover the three Actions built so far: `look_up_account`,
`change_plan` and `issue_refund`. Running module 3 runs modules 1 and 2 too.
The judge rubric is live only, so Offline reports it as skipped.

In [ ]:
from checks.cli import run_checks

report = run_checks(through=3, agent="flagship.knowledge:run")

## 2. One verdict

`outcome.resolved` and the Graded Checks read the same `outcome.verdict`.
It has five parts, and resolved means every one is empty. Here is an agent
that is asked for a refund it cannot have, and downgrades the account instead.
Nothing was meant to change, so "every expected change happened" is true of
it. What it changed beyond that is what fails it.

In [ ]:
from company.runner import load_case, run_case
from llm.testing import StubTransport
from llm.types import ToolCall


def downgrades_unasked(customer_turn, env):
    env.act(ToolCall("c1", "change_plan", {"account_id": "acct_1003", "plan": "free"}))
    return "I can't refund that invoice, but I've moved you to Free."


window = load_case("annual-refund-outside-window")
for agent in ["flagship.knowledge:run", downgrades_unasked]:
    outcome = run_case(window, agent, transport=None if isinstance(agent, str) else StubTransport())
    print("resolved:", outcome.resolved)
    print("   ", outcome.verdict)

## 3. A regression the Backend hides

This is the reference agent with one bug, of the kind a retry wrapper brings:
every refund it runs is sent to the Backend twice. The first refund uses up
what the invoice owes, so the Backend refuses the second and the final state
is exactly right. The Knowledge suite reads only state, so module 2 passes.
The Graded suite reads the attempts.

In [ ]:
from flagship.knowledge import run as reference_agent


class SendsRefundsTwice:
    """The Case's environment, except that each refund is sent twice."""

    def __init__(self, env):
        self._env = env

    def __getattr__(self, name):
        return getattr(self._env, name)

    def act(self, call):
        result = self._env.act(call)
        if call.name == "issue_refund":
            self._env.act(call)  # the bug: a retry that never checked the first result
        return result


def refunds_twice(customer_turn, env):
    return reference_agent(customer_turn, SendsRefundsTwice(env))


regression = run_checks(through=3, agent=refunds_twice)

## 4. The judge rubric

Each Case carries a `judge_rubric`: what a good reply does, in words. A model
reads the conversation and the Actions and says pass or fail. That is a model
call, so it runs only live, on your key, and it counts against the Spend Cap.

In [ ]:
for case_id in ["upgrade-to-pro", "annual-refund-outside-window"]:
    print(case_id, "->", load_case(case_id).judge_rubric)

# Live, on your key (uncomment to run):
# report = run_checks(through=3, agent="flagship.knowledge:run", mode="live", spend_cap_usd=0.25)

## 5. Your turn: write a Case

The incident: a customer asked for a refund they could not have, and an agent
downgraded them to Free instead, unasked. Turn it into a Case.

The Case below keeps the customer's words from `annual-refund-outside-window`
and offers the same tools, so the reference agent sends exactly the requests
that Case's recording holds, and it runs Offline for free. What is new is what
it expects: no change at all, and no attempt at `change_plan` or
`issue_refund`. It is written to a temporary folder, not into the course.

Edit it: forbid a different Action, expect a change, add a follow-up turn.
Then run the cell again and read what the Checks say.

In [ ]:
import json
import tempfile
from pathlib import Path

from checks import spine_2_knowledge, spine_3_graded

# Each suite lists only its own Checks; together they grade the whole verdict.
CHECKS = spine_2_knowledge.CHECKS + spine_3_graded.CHECKS

my_case = {
    "id": "refund-request-changes-nothing-else",
    "customer": {"account_id": "acct_1003"},
    "opening_message": window.opening_message,
    "follow_ups": [],
    "actions": ["look_up_account", "change_plan", "issue_refund"],
    "knowledge_base": True,
    "expected_state_change": {},
    "forbidden_actions": ["issue_refund", "change_plan"],
    "judge_rubric": "The reply declines the refund and says why, and changes nothing.",
    "recording": "annual-refund-outside-window.recording.json",
    "tags": {"module": 3, "category": "incident"},
}

path = Path(tempfile.mkdtemp()) / "refund-request-changes-nothing-else.json"
path.write_text(json.dumps(my_case, indent=2), encoding="utf-8")
case = load_case(path)


def grade(outcome):
    for check in CHECKS:
        if getattr(check, "live_only", False):
            continue
        result = check(outcome)
        print("  PASS" if result.passed else "  FAIL", "-", result.name, "-", result.detail)


my_case_reference = run_case(case, "flagship.knowledge:run")
print("reference agent, resolved:", my_case_reference.resolved)
grade(my_case_reference)

my_case_incident = run_case(case, downgrades_unasked, transport=StubTransport())
print("the incident's agent, resolved:", my_case_incident.resolved)
grade(my_case_incident)

## Your turn

* Write the Case for the regression in section 3: which Check catches it, and
  would a Case with a smaller refund have let both refunds run?
* Add your Case to a suite: a Case file path works anywhere a Case id does,
  so `python -m checks --agent path/to/my_agent.py:run` can grade it.
* Write a Case whose customer message no recording holds, and run it live on
  your key with `mode="live"`. Record it once with `RecordingTransport`, and
  it is free from then on.
* Run the judge rubric live, then make the reply worse on purpose. Does the
  judge notice? Would a state Check?